# SMILES Standardization and Descriptor Calculation Tutorial
SMILES (Simplified Molecular Input Line Entry System) is a text-based notation for representing chemical structures. It's like a "molecular barcode" that describes how atoms are connected.

Examples: Water: O ; Methane: C ; Benzene: c1ccccc1 or C1=CC=CC=C1 ; Ethanol: CCO or OCC

This notebook demonstrates how to:
1. Standardize SMILES strings using RDKit
2. Calculate molecular descriptors
3. Generate polymer-specific descriptors

## Section 1: Import Required Libraries

Let's start by importing all the necessary libraries for cheminformatics work.

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# RDKit for cheminformatics
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

## SMILES Standardization with RDKit

Let's see how RDKit canonicalizes SMILES strings. Canonicalization ensures that different representations of the same molecule are converted to a standard form. This is needed because same molecule can be written in many different ways. This creates problems for:

- Database searches: Can't find duplicates
- Machine learning: Same molecule treated as different
- Descriptor calculation: Inconsistent results
- Data analysis: Artificial data inflation

RDKit uses sophisticated algorithms to:

1. Assign unique atom labels based on connectivity
2. Order atoms systematically (heavy atoms first, then by connectivity)
3. Handle aromaticity consistently (lowercase = aromatic)
4. Resolve tie-breaking using chemical rules


In [2]:
# Demonstrate canonicalization
print("🔄 SMILES Canonicalization Examples, they all are the same molecule")

smiles_variants = [
    'CCc1ccccc1',     # Standard way
    'c1ccccc1CC',     # Starting from benzene
    'C(c1ccccc1)C',   # Parentheses grouping
    'c1cc(CC)ccc1',   # Different substitution position
]

for smiles in smiles_variants:
    mol = Chem.MolFromSmiles(smiles)
    canonical_smiles = Chem.MolToSmiles(
        mol
    )
    print(f"Original:   {smiles}")
    print(f"Canonical:  {canonical_smiles}")
    print("-" * 30)

🔄 SMILES Canonicalization Examples, they all are the same molecule
Original:   CCc1ccccc1
Canonical:  CCc1ccccc1
------------------------------
Original:   c1ccccc1CC
Canonical:  CCc1ccccc1
------------------------------
Original:   C(c1ccccc1)C
Canonical:  CCc1ccccc1
------------------------------
Original:   c1cc(CC)ccc1
Canonical:  CCc1ccccc1
------------------------------


## Remove Salts and Fragments
### What are Salts and Fragments in SMILES?
Salts and fragments occur when SMILES strings contain multiple disconnected molecular components separated by dots (.). This commonly happens in chemical databases where:

 - Salts: Ionic compounds with separate cation and anion parts
 - Solvates: Molecules with associated solvent molecules
 - Mixtures: Multiple distinct molecules in one entry
 - Counterions: Charged species that balance ionic compounds

For polymer property prediction, we need **clean, single-component structures** because:
- Descriptor calculation should focus on the main polymer structure
- Salt components don't directly affect polymer backbone properties
- Multiple fragments create ambiguity about which structure to analyze
- Standardization requires consistent single-molecule representations

RDKit can automatically remove salts and select the largest fragment from SMILES containing multiple components.

In [3]:
# Examples with salts and fragments
salt_examples = [
    'CCO.Cl',           # Ethanol with chloride
    'CC(=O)O.[Na+]',    # Sodium acetate
    'CC.O.CC',          # Multiple fragments
    'c1ccccc1.CCO',     # Benzene + ethanol
]

print("🧂 Salt Removal Examples:")
print("=" * 35)

for smiles in salt_examples:
    mol = Chem.MolFromSmiles(smiles)
    
    # Remove salts
    mol_no_salt = SaltRemover().StripMol(mol)
    
    # Convert back to canonical SMILES
    cleaned = Chem.MolToSmiles(
        mol, # molecule object
        canonical=True # ensure canonical form
    )
    
    print(f"Original:  {smiles}")
    print(f"Cleaned:   {cleaned}")
    print("-" * 25)

🧂 Salt Removal Examples:
Original:  CCO.Cl
Cleaned:   CCO.Cl
-------------------------
Original:  CC(=O)O.[Na+]
Cleaned:   CC(=O)O.[Na+]
-------------------------
Original:  CC.O.CC
Cleaned:   CC.CC.O
-------------------------
Original:  c1ccccc1.CCO
Cleaned:   CCO.c1ccccc1
-------------------------
